# NeMo Agent Toolkit with the Movie MCP Server

Connect the movie database MCP server from Notebook 2 to a Nemotron-powered ReAct agent, ask natural-language movie questions, and inspect the workflow and function spans that NeMo Agent Toolkit (NAT) exports to Arize Phoenix.

Run `02_movie_database_mcp.ipynb` first. It creates `movie_db.py` and `movie_server.py`; this notebook launches the server and connects to it.

## Setup

This notebook accepts either `NVIDIA_API_KEY` or `NGC_API_KEY` from the environment. Credentials are never written to the generated YAML files.

In [ ]:
import os
from getpass import getpass

if not os.environ.get("NVIDIA_API_KEY"):
    if os.environ.get("NGC_API_KEY"):
        os.environ["NVIDIA_API_KEY"] = os.environ["NGC_API_KEY"]
    else:
        os.environ["NVIDIA_API_KEY"] = getpass("Enter your NVIDIA API Key: ").strip()

## Start the Movie MCP Server

Launch the high-level FastMCP server generated by Notebook 2 and give it a few seconds to start. The server uses `MCP_PORT` configured in `.vscode/notebook.env`.

In [ ]:
import subprocess
import time

MCP_PORT = int(os.environ["MCP_PORT"])

# Start the HTTP server in the background
mcp_server_process = subprocess.Popen(
    ["python", "movie_server.py", "data/movie.sqlite"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
)

# Wait for the server to start
time.sleep(3)

SERVER_PID = mcp_server_process.pid
MCP_URL = f"http://127.0.0.1:{MCP_PORT}/mcp"
print(f"MCP server started on {MCP_URL} (PID: {SERVER_PID})")

## Define a NAT Workflow

NeMo Agent Toolkit workflows are defined in YAML. This workflow has three main sections:

- `function_groups`: the external movie MCP server;
- `llms`: the Nemotron model hosted by NVIDIA NIM; and
- `workflow`: a ReAct agent that can call the movie tools.

`parse_agent_response_max_retries: 3` allows the agent to recover when an LLM response does not follow the ReAct format.

In [ ]:
%%writefile movie_workflow.yml
function_groups:
  mcp_movies:
    _type: mcp_client
    server:
      transport: streamable-http
      url: "http://127.0.0.1:${MCP_PORT}/mcp"

llms:
  nim_llm:
    _type: nim
    model_name: nvidia/nemotron-3-super-120b-a12b
    base_url: https://integrate.api.nvidia.com/v1
    api_key: ${NVIDIA_API_KEY}
    temperature: 0.0
    max_tokens: 1024

workflow:
  _type: react_agent
  tool_names:
    - mcp_movies
  llm_name: nim_llm
  verbose: true
  parse_agent_response_max_retries: 3


Run the workflow once without tracing as a baseline. The agent should call `search_movies` and summarize the returned rows.

In [ ]:
!nat run --config_file movie_workflow.yml --input "movies rated over 8.5"

## Start the Phoenix Observability Server

NAT's built-in Phoenix exporter records the intermediate events that NAT emits as workflow and function spans. For this workflow, the useful span is the `mcp_movies__search_movies` function span: its input contains the tool arguments, and its output contains the rows returned by the MCP tool. The exporter does not automatically create separate spans for the NIM request, raw MCP transport, MCP server internals, or SQLite query. The bootcamp environment forwards port 6006 to the same port on your local machine.

In [ ]:
import os
import subprocess
import time

PHOENIX_PORT = int(os.environ.get("PHOENIX_PORT", "6006"))
os.environ["PHOENIX_PORT"] = str(PHOENIX_PORT)

phoenix_log = open("phoenix.log", "w")
phoenix_process = subprocess.Popen(
    ["phoenix", "serve"],
    stdout=phoenix_log,
    stderr=subprocess.STDOUT,
)

time.sleep(3)

print(f"Server started on http://127.0.0.1:{PHOENIX_PORT} (PID: {phoenix_process.pid})")
print(f"Access via port-forwarding at http://localhost:{PHOENIX_PORT}")
print("Logs: phoenix.log")

### Reading Traces

After the traced workflow runs, open the `movie-mcp` project in Phoenix. With NAT 1.8, this example typically produces workflow spans plus a `mcp_movies__search_movies` function span. Select that function span and inspect its **Input** and **Output** fields; the output is where the retrieved movie JSON appears. The exact span set can vary by NAT version and plugin.

- **Wrong tool arguments:** inspect the function span input.
- **Wrong retrieved data:** inspect the function span output before comparing it with the final answer.
- **Slow workflow or tool:** compare the durations of the spans that are present.
- **Tool errors:** inspect the function span status and recorded exception attributes.

A missing NIM, MCP transport, server, or SQL span does not mean that operation did not occur; it means this built-in exporter did not create a separate span for it.

## Configure Phoenix Tracing

Telemetry is configured in YAML. This enables NAT's Phoenix exporter for events emitted by the workflow; it does not automatically instrument every library or process involved in the request.

In [ ]:
%%writefile movie_workflow_tracing.yml
general:
  telemetry:
    tracing:
      phoenix:
        _type: phoenix
        endpoint: http://127.0.0.1:${PHOENIX_PORT}/v1/traces
        project: movie-mcp

function_groups:
  mcp_movies:
    _type: mcp_client
    server:
      transport: streamable-http
      url: "http://127.0.0.1:${MCP_PORT}/mcp"

llms:
  nim_llm:
    _type: nim
    model_name: nvidia/nemotron-3-super-120b-a12b
    base_url: https://integrate.api.nvidia.com/v1
    api_key: ${NVIDIA_API_KEY}
    temperature: 0.0
    max_tokens: 1024

workflow:
  _type: react_agent
  tool_names:
    - mcp_movies
  llm_name: nim_llm
  verbose: true
  parse_agent_response_max_retries: 3


Run a specific query and then inspect its trace in Phoenix:

In [ ]:
!nat run --config_file movie_workflow_tracing.yml --input "What is the rating of the movie The Dark Knight Rises?"

### Review the Trace

Open the Phoenix URL printed above, select the `movie-mcp` project, and inspect the latest trace. In NAT 1.8, look for the CHAIN span named `mcp_movies__search_movies`, which represents the function call. Its input should include the title filter, and its output should contain the JSON rows returned by the tool. You may see only a small span tree containing workflow and function spans; separate LLM, MCP transport, server execution, and SQLite spans are outside the tracing coverage demonstrated in this notebook.

> **Instrumentation boundary:** richer distributed traces require additional OpenTelemetry/OpenInference instrumentation in both the NAT process and the separately launched MCP server. That setup is intentionally not claimed or demonstrated here.

## Optional Challenge: Cover All Tables

The current tool searches only six columns in the `IMDB` table. Extend `MovieDB` and the FastMCP server to answer questions that require the `earning` and `genre` tables or additional `IMDB` columns.

| Question | What it exercises |
| --- | --- |
| What's the worldwide gross of Inception? | `IMDB` and `earning` |
| Which biography has the highest IMDb rating? | `IMDB` and `genre` |
| What's the longest movie rated 8.5 or higher? | `Runtime` and `Rating` |
| Which movie has the biggest rating gap between male and female viewers? | `VotesM` and `VotesF` |
| Which movie is most loved by viewers under 18? | `VotesU18` |
| Which movie has the biggest disagreement between critics and audiences? | `MetaCritic` and `Rating` |
| Which comedy had the best return on investment? | All three tables and arithmetic |

After adding tools, restart the MCP process, verify them with `nat mcp client tool list`, rerun a workflow question, and inspect any function spans that NAT exports to Phoenix.

## Cleanup

Stop both background processes when you finish.

In [ ]:
for name, process in [
    ("MCP server", mcp_server_process),
    ("Phoenix", phoenix_process),
]:
    process.terminate()
    try:
        process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait(timeout=5)
    print(f"{name} (PID: {process.pid}) stopped")

phoenix_log.close()

## Links and Resources

- **[NVIDIA NeMo Agent Toolkit Overview](https://docs.nvidia.com/nemo/agent-toolkit/latest/)**: Core toolkit concepts, installation guidance, and workflow examples.
- **[NeMo Agent Toolkit Workflow Configuration](https://docs.nvidia.com/nemo/agent-toolkit/latest/build-workflows/workflow-configuration.html)**: Reference documentation for YAML sections, ReAct workflows, and environment-variable interpolation.
- **[NeMo Agent Toolkit as an MCP Client](https://docs.nvidia.com/nemo/agent-toolkit/latest/build-workflows/mcp-client.html)**: Guidance for discovering MCP tools and registering them as workflow functions.
- **[Observe NeMo Agent Toolkit Workflows](https://docs.nvidia.com/nemo/agent-toolkit/latest/workflows/observe/index.html)**: An overview of event-driven observability and telemetry exporter configuration.
- **[Phoenix Traces and Spans](https://arize.com/docs/phoenix/tracing/concepts-tracing/what-are-traces)**: An introduction to traces, spans, span kinds, attributes, and projects.
- **[Agentic AI Bootcamp: NeMo Agent Toolkit Notebook](https://github.com/openhackathons-org/agentic-ai-bootcamp/blob/main/tutorial/jupyter_notebook/05_nemo_agent_toolkit.ipynb)**: A related end-to-end workshop implementation for additional practice.

---

## Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.